# K_03 – Marktdynamik

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt (Kür)

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Patrik Neunteufel | **Datum:** März 2026

---

*Spread-Entwicklung 2018–2024 und [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex)-Lernkurven-Projektion bis 2035.*


| [← K_02 – Cross-Border-Analyse](K_02_Cross_Border.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [K_04 – Saisonale Animationen →](K_04_Animationen.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_03'></a>

[Einleitung](#einleitung_K_03)  
[Initialisierung](#initialisierung_K_03)  
1 [Analyse: Spread-Entwicklung](#analyse-spread-entwicklung_K_03)  
2 [Chart K03-A: Spread- & Volatilitäts-Zeitreihe](#chart-k03-a-spread-volatilitaets-zeitreihe_K_03)  
3 [Analyse: CAPEX-Lernkurve](#analyse-capex-lernkurve_K_03)  
4 [Chart K03-B: CAPEX × Spread Sensitivitäts-Heatmap](#chart-k03-b-capex-spread-sensitivitaets-heatmap_K_03)  
5 [Synthese: Wann kippt der Business Case?](#synthese-wann-kippt-der-business-case_K_03)  
[Fazit](#fazit_K_03)  
[Abschluss](#abschluss_K_03)  


---
## Einleitung <a id='einleitung_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

Marktdynamik: Wie hat sich der Preis-Spread historisch entwickelt, und wann
kippt der Business Case durch die Kombination von sinkenden CAPEX-Kosten
und steigendem Spread?

- **Spread-Zeitreihe** rollierend (12-Monate) auf NB02-Preisdaten
- **CAPEX-Lernkurve** mit Projektion bis 2035 (angenommen −10%/Jahr Lernrate)
- **Sensitivitäts-Heatmap** CAPEX × Spread → Break-Even-Linien
- **Monitoring-Trigger** für Investitionsentscheidung: ab wann lohnt sich der
  Einstieg je Segment?


## Initialisierung<a id='initialisierung_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

Bibliotheken laden, `../sync/config.json` lesen, Verzeichnispfade setzen.

**Imports und Versionen:**

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, warnings, json
import numpy  as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')
from datetime import datetime

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json` (SSOT), setzt Pfade.

In [ ]:
with open('../sync/config.json') as _f:
    CFG = json.load(_f)

MODE          = CFG['mode']
DIR_RAW       = os.path.join('../data', 'raw')
DIR_PROCESSED = os.path.join('../data', 'processed')
DIR_INTER     = os.path.join('../data', 'intermediate')
SZ_AKTIV      = CFG['szenarien']['gleichzeitigkeit_aktiv']
FORCE_RELOAD = CFG.get('force_reload', {})  # konventionskonform gelesen
DIR_INTER_SZ  = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR       = os.path.join('../output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: ../sync/config.json

# ── Farben & Stil aus ../sync/config.json (SSOT) ─────────────────────────────────────
# Bestehende Variablen (Rückwärtskompatibilität)
_viz        = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK     = _viz.get('bg_dark',    '#0d1117')
BG_PANEL    = _viz.get('bg_panel',   '#141414')
C_PRICE     = _viz.get('c_price',    '#FFA726')
C_LOAD      = _viz.get('c_load',     '#66BB6A')
C_CHARGE    = _viz.get('c_charge',   '#1565C0')
C_FEED      = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS  = _viz.get('seg_colors', ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS

# UI-Strukturfarben
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')  # Achsenbeschriftungen
C_TICK       = _viz.get('c_tick',       '#bbbbbb')  # Tick-Labels
C_SPINE      = _viz.get('c_spine',      '#333333')  # Achsenrahmen
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')  # Legenden-Hintergrund
C_GITTER     = _viz.get('c_gitter',     '#cccccc')  # Gitterlinien

# Funktionale Extrafarben (nur laden was das NB braucht)
C_DISPATCH   = _viz.get('c_dispatch',   '#AB47BC')  # Dispatch-optimal
C_STACKING   = _viz.get('c_stacking',   '#5DCAA5')  # Revenue Stacking
C_SOLAR      = _viz.get('c_solar',      '#FDD835')  # Solar-Ertrag
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')  # Grenzwert / Warnung
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')  # Flusswasser / Alt. Speicher
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')  # Erneuerbare

# Stilkonstanten
_stil               = CFG.get('visualisierung', {}).get('stil', {})
LW                  = _stil.get('linienbreite_standard', 1.5)   # Standard-Linienbreite
LW_DUENN            = _stil.get('linienbreite_duenn',    0.8)   # dünne Linien
LW_DICK             = _stil.get('linienbreite_dick',     2.5)   # dicke Linien
ALPHA_FLAECHE       = _stil.get('alpha_flaeche',         0.12)  # dezente Füllung
ALPHA_FLAECHE_STARK = _stil.get('alpha_flaeche_stark',   0.35)  # Balken / Füllung
ALPHA_LEGENDE       = _stil.get('alpha_legende',         0.30)  # Legenden-BG
ALPHA_GEDAEMPFT     = _stil.get('alpha_linie_gedaempft', 0.55)  # Nebenlinien
FS_TITEL            = _stil.get('schriftgroesse_titel',   13)   # Chart-Titel
FS_ACHSE            = _stil.get('schriftgroesse_achse',   10)   # Achsenbeschr.
FS_TICK             = _stil.get('schriftgroesse_tick',     9)   # Ticks
FS_LEGENDE          = _stil.get('schriftgroesse_legende',  8)   # Legende
FS_KLEIN            = _stil.get('schriftgroesse_klein',    7)   # Annotationen

# matplotlib rcParams — nur stabile, versionsunabhängige Keys (matplotlib >= 3.5)
# axes.titlecolor (3.8+) und axes.grid (stört Karten) bewusst NICHT gesetzt

mpl.rcParams.update({
    'figure.facecolor':  BG_DARK,
    'axes.facecolor':    BG_PANEL,
    'axes.edgecolor':    C_SPINE,
    'axes.labelcolor':   C_ACHSE,
    'axes.labelsize':    FS_ACHSE,
    'axes.titlesize':    FS_TITEL,
    'xtick.color':       C_TICK,
    'ytick.color':       C_TICK,
    'xtick.labelsize':   FS_TICK,
    'ytick.labelsize':   FS_TICK,
    'text.color':        'white',
    'lines.linewidth':   LW,
    'legend.facecolor':  C_LEGENDE_BG,
    'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize':   FS_LEGENDE,
    'legend.edgecolor':  C_SPINE,
})
print('Farben & Stil geladen.')

# Marktdynamik-Parameter aus ../sync/config.json
_md = CFG['kuer']['markt']
SPREAD_TRIGGER_EUR = _md['spread_breakeven_privat_eur_mwh']   # 30
CAPEX_LERNRATE     = _md['capex_lernrate_pct_pro_jahr']        # 10
CAPEX_TRIGGER      = _md['capex_ziel_privat_eur_kwh']          # 250

print(f'MODE           : {MODE}')
print(f'Spread-Trigger : > {SPREAD_TRIGGER_EUR} EUR/MWh Monatsmedian')
print(f'CAPEX-Lernrate : -{CAPEX_LERNRATE}%/Jahr')
print(f'CAPEX-Trigger  : < {CAPEX_TRIGGER} EUR/kWh installiert')

**Daten laden:** Spread-Zeitreihe aus `intermediate/` und Import/Export-Analyse
aus K_02 — beide Dateien werden mit Existenzprüfung geladen.


In [ ]:
# ── Daten laden ───────────────────────────────────────────────────────────────
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')
ECON_FILE   = os.path.join(DIR_INTER_SZ, 'wirtschaftlichkeit.csv')

df_sp   = None  # immer initialisieren
df_econ = None

if not os.path.exists(SPREAD_FILE):
    print('⚠️  spread_zeitreihe.csv fehlt → NB02 Sektion 7 ausführen.')
else:
    df_sp = pd.read_csv(SPREAD_FILE, parse_dates=['yearmonth'])

df_econ = pd.read_csv(ECON_FILE) if os.path.exists(ECON_FILE) else None

if df_sp is not None:
    import numpy as np
    print(f'Spread-Zeitreihe : {df_sp.shape} | {df_sp["yearmonth"].min().strftime("%Y-%m")} – {df_sp["yearmonth"].max().strftime("%Y-%m")}')
    print(f'Ø Spread         : {df_sp["spread_median"].mean():.1f} EUR/MWh')
    z = np.polyfit(range(len(df_sp)), df_sp['spread_median'], 1)
    print(f'Trend            : {z[0]:+.3f} EUR/MWh/Monat ({z[0]*12:+.2f} EUR/MWh/Jahr)')
    print(f'Negativpreis-Std : {df_sp["neg_price_h"].sum():,} total ({df_sp["neg_price_h"].mean():.1f}/Monat)')
    display(df_sp.head(3))
else:
    print('df_sp nicht verfügbar — nachfolgende Zellen überspringen.')


---
## 1. Analyse: Spread-Entwicklung <a id='analyse-spread-entwicklung_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

**Ausgangslage** (aus NB07_Erweiterungen Sektion 2, jetzt hier vertieft):

Der Intra-Tag-Spread ist der direkteste [ROI](../organisation/O_02_Glossar.ipynb#g-roi)-Treiber — jeder EUR/MWh mehr Spread  
bedeutet mehr Erlös pro [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Zyklus. 2023/2024 lagen die CH-Spreads historisch  
tief (normalisierte Post-Krisen-Preise nach dem Energiekrisen-Peak 2022).

Mit steigendem Anteil erneuerbarer Energien (Solar, Wind) steigt die strukturelle  
[Volatilität](../organisation/O_02_Glossar.ipynb#g-volatilitaet): Mittags-Tief durch Solar, Abend-Spitze durch Nachfrage. Dieser Effekt  
ist in DE/AT bereits messbar und wird in der CH mit dem Solarausbau folgen.

### Break-Even-Tabelle nach Spread

| Spread [EUR/MWh] | Break-Even Privat (10kWh) | Bewertung |
|---|---|---|
| < 15 | > 30 Jahre | Nicht rentabel |
| 20–30 | 15–25 Jahre | Grenzwertig |
| 30–50 | 8–15 Jahre | Mit [VPP](../organisation/O_02_Glossar.ipynb#g-vpp) attraktiv |
| > 50 | < 8 Jahre | Klar rentabel |

**Monitoring-Trigger:** Investition lohnt sich wenn CH-Monatsmedian-Spread  
**dauerhaft > 30 EUR/MWh** — Schwellenwert in `../sync/config.json → kuer.markt.spread_breakeven_privat_eur_mwh`.


In [ ]:
if df_sp is None:
    print('⚠️  Übersprungen — Spread-Daten fehlen.')
else:
    # ── Spread-Analyse: Trendberechnung & Monitoring ─────────────────────────────
    xn = np.arange(len(df_sp))
    z  = np.polyfit(xn, df_sp['spread_median'], 1)
    trend_j = z[0] * 12  # EUR/MWh pro Jahr
    
    # Extrapolation bis Trigger
    current = df_sp['spread_median'].mean()
    if trend_j > 0:
        jahre_bis_trigger = (SPREAD_TRIGGER_EUR - current) / trend_j
        print(f'Aktueller Ø Spread : {current:.1f} EUR/MWh')
        print(f'Trend              : {trend_j:+.2f} EUR/MWh/Jahr')
        print(f'Trigger ({SPREAD_TRIGGER_EUR} EUR/MWh): in ~{jahre_bis_trigger:.0f} Jahren (falls Trend hält)')
    else:
        print(f'Trend negativ ({trend_j:.2f}/Jahr) — Trigger unklar')
    
    # Monate über/unter Trigger
    über = (df_sp['spread_median'] >= SPREAD_TRIGGER_EUR).mean() * 100
    print(f'Monate >= {SPREAD_TRIGGER_EUR} EUR/MWh: {über:.0f}% der Datensätze')
    


---
## 2. Chart K03-A: Spread- & Volatilitäts-Zeitreihe <a id='chart-k03-a-spread-volatilitaets-zeitreihe_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

*Direkt übernommen aus NB03 Chart 7 — hier eingebettet für eigenständige Ausführbarkeit.*  
Trendlinie zeigt ob das Arbitrage-Potenzial strukturell steigt oder fällt.


In [ ]:
# ── Chart 7: Spread- & Volatilitäts-Zeitreihe ───────────────────────────────
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')  # aus NB02 Sektion 7

if not os.path.exists(SPREAD_FILE):
    print('spread_zeitreihe.csv nicht vorhanden → NB02 Sektion 6 ausführen.')
else:
    df_sp = pd.read_csv(SPREAD_FILE, parse_dates=['yearmonth'])
    x  = df_sp['yearmonth']
    xn = np.arange(len(df_sp))
    z  = np.polyfit(xn, df_sp['spread_median'], 1)

    # ── Gesamtchart (3 Panels) ────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
    fig.patch.set_facecolor(BG_DARK)
    for ax in axes:
        ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
        for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
    fig.suptitle('Arbitrage-Potenzial Entwicklung — Monatlicher Spread & Volatilität',
                 color='white', fontsize=FS_TITEL, fontweight='bold')

    ax = axes[0]
    ax.fill_between(x, df_sp['spread_median'], alpha=0.3, color=C_PRICE)
    ax.plot(x, df_sp['spread_median'], color=C_PRICE, lw=2)
    ax.axhline(df_sp['spread_median'].mean(), color='white', lw=1, linestyle='--', alpha=0.5,
               label=f'Ø {df_sp["spread_median"].mean():.1f} EUR/MWh')
    ax.plot(x, np.polyval(z, xn), color=C_UTIL, lw=LW, linestyle=':',
            label=f'Trend ({z[0]:+.2f} EUR/MWh/Monat)')
    ax.set_ylabel('Spread [EUR/MWh]', color=C_ACHSE)
    ax.set_title('Intra-Tag-Spread (p75−p25 Median)', color='white')
    ax.legend(fontsize=FS_LEGENDE, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
    ax.grid(True, alpha=0.10)

    ax = axes[1]
    ax.fill_between(x, df_sp['volatility'], alpha=0.3, color=C_PRIV)
    ax.plot(x, df_sp['volatility'], color=C_PRIV, lw=2)
    ax.axhline(df_sp['volatility'].mean(), color='white', lw=1, linestyle='--', alpha=0.5,
               label=f'Ø {df_sp["volatility"].mean():.1f} EUR/MWh')
    ax.set_ylabel('Volatilität [EUR/MWh]', color=C_ACHSE)
    ax.set_title('Marktvolatilität (Std. Abweichung Stundenpreise)', color='white')
    ax.legend(fontsize=FS_LEGENDE, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
    ax.grid(True, alpha=0.10)

    ax = axes[2]
    colors_neg = [C_UTIL if v > 0 else '#444' for v in df_sp['neg_price_h']]
    ax.bar(x, df_sp['neg_price_h'], color=colors_neg, alpha=0.85, width=25,
           label='Stunden mit Preis < 0 EUR/MWh')
    ax.set_ylabel('Negativpreis-Stunden', color=C_ACHSE)
    ax.set_title('Negativpreise pro Monat (Solar-Überschuss-Indikator)', color='white')
    ax.legend(fontsize=FS_LEGENDE, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
    ax.grid(True, axis='y', alpha=0.10)
    ax.set_xlabel('Monat', color=C_ACHSE)

    plt.tight_layout()
    p7 = os.path.join(CHARTS_DIR, 'kuer_k03_spread_zeitreihe.png')
    plt.savefig(p7, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
    plt.show(); plt.close()
    trend_dir = 'steigend' if z[0] > 0 else 'fallend'
    print(f'Chart 7 gespeichert: {p7}')
    print(f'Trend: Spread {trend_dir} ({z[0]:+.3f} EUR/MWh/Monat = {z[0]*12:+.2f}/Jahr)')

    # ── Einzelplot 1: Intra-Tag-Spread mit Trend ──────────────────────────────
    for fname, col, title, ylabel, data_col, with_trend in [
        ('chart7a_spread_trend.png',  C_PRICE, 'Intra-Tag-Spread (p75−p25 Median)',
         'Spread [EUR/MWh]', 'spread_median', True),
        ('chart7b_volatilitaet.png',  C_PRIV, 'Marktvolatilität (Std. Abweichung)',
         'Volatilität [EUR/MWh]', 'volatility', False),
    ]:
        fig_e, ax_e = plt.subplots(figsize=(12, 4))
        fig_e.patch.set_facecolor(BG_DARK); ax_e.set_facecolor(BG_PANEL)
        ax_e.tick_params(colors=C_TICK)
        for sp in ax_e.spines.values(): sp.set_edgecolor(C_SPINE)
        ax_e.fill_between(x, df_sp[data_col], alpha=0.3, color=col)
        ax_e.plot(x, df_sp[data_col], color=col, lw=2,
                  label=f'Ø {df_sp[data_col].mean():.1f} EUR/MWh')
        ax_e.axhline(df_sp[data_col].mean(), color='white', lw=1, linestyle='--', alpha=0.5)
        if with_trend:
            ax_e.plot(x, np.polyval(z, xn), color=C_UTIL, lw=LW, linestyle=':',
                      label=f'Trend ({z[0]:+.2f} EUR/MWh/Monat)')
        ax_e.set_title(title, color='white', fontsize=12)
        ax_e.set_ylabel(ylabel, color=C_ACHSE)
        ax_e.set_xlabel('Monat', color=C_ACHSE)
        ax_e.legend(fontsize=FS_TICK, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
        ax_e.grid(True, alpha=0.10)
        plt.tight_layout()
        plt.savefig(os.path.join(CHARTS_DIR, fname), dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
        plt.close(); print(f'  Einzelplot: {fname}')

    # ── Einzelplot 3: Negativpreise ───────────────────────────────────────────
    fig_e, ax_e = plt.subplots(figsize=(12, 4))
    fig_e.patch.set_facecolor(BG_DARK); ax_e.set_facecolor(BG_PANEL)
    ax_e.tick_params(colors=C_TICK)
    for sp in ax_e.spines.values(): sp.set_edgecolor(C_SPINE)
    colors_neg = [C_UTIL if v > 0 else '#444' for v in df_sp['neg_price_h']]
    ax_e.bar(x, df_sp['neg_price_h'], color=colors_neg, alpha=0.85, width=25,
             label='Stunden mit Preis < 0 EUR/MWh')
    ax_e.set_title('Negativpreise pro Monat (Solar-Überschuss-Indikator)', color='white', fontsize=12)
    ax_e.set_ylabel('Negativpreis-Stunden', color=C_ACHSE)
    ax_e.set_xlabel('Monat', color=C_ACHSE)
    ax_e.legend(fontsize=FS_TICK, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white')
    ax_e.grid(True, axis='y', alpha=0.10)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_DIR, 'kuer_k03_negativpreise.png'),
                dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
    plt.close(); print('  Einzelplot: kuer_k03_negativpreise.png')


---
## 3. Analyse: CAPEX-Lernkurve <a id='analyse-capex-lernkurve_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

**Historische Entwicklung** (Quelle: BNEF / IEA, aus NB07_Erweiterungen Sektion 3):

| Jahr | [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) Cell [USD/kWh] | Systempreis CH (est.) |
|---|---|---|
| 2015 | ~350 | ~780 EUR/kWh |
| 2020 | ~140 | ~580 EUR/kWh |
| 2023 | ~90–130 | ~390–580 EUR/kWh |
| 2030 (Proj.) | ~60–80 | ~240–340 EUR/kWh |

*Systempreis = Cell + Wechselrichter + Installation + Montage (Faktor ~2–3×)*

### Projektion bei −10%/Jahr Lernrate

| Jahr | CAPEX Privat (Schätzung) | Break-Even bei Spread 25 EUR/MWh |
|---|---|---|
| 2025 | ~350 EUR/kWh | > 25 Jahre |
| 2028 | ~260 EUR/kWh | ~18 Jahre |
| 2031 | ~190 EUR/kWh | ~13 Jahre |
| 2034 | ~140 EUR/kWh | ~9 Jahre |

**Monitoring-Trigger:** Privat-Investition attraktiv wenn Systempreis  
**< 250 EUR/kWh installiert** — Schwellenwert in `../sync/config.json → kuer.markt.capex_ziel_privat_eur_kwh`.


In [ ]:
# ── CAPEX-Lernkurven-Projektion ───────────────────────────────────────────────
basisjahr    = 2025
capex_basis  = 350   # EUR/kWh (Systempreis 2025, Schätzung)
lernrate     = CAPEX_LERNRATE / 100

jahre    = list(range(basisjahr, basisjahr + 16))
capex_pr = [capex_basis * (1 - lernrate)**i for i in range(len(jahre))]

print(f'CAPEX-Projektion ({CAPEX_LERNRATE:.0f}%/Jahr Lernrate):')
print(f'  {"Jahr":<6} {"CAPEX [EUR/kWh]":>16} {"Status":>30}')
for j, c in zip(jahre, capex_pr):
    status = '← TRIGGER UNTERSCHRITTEN' if c < CAPEX_TRIGGER and capex_pr[jahre.index(j)-1] >= CAPEX_TRIGGER else ''
    marker = '>>>' if status else '   '
    print(f'  {marker} {j:<4} {c:>14.0f}  {status}')



---
## 4. Chart K03-B: CAPEX × Spread Sensitivitäts-Heatmap <a id='chart-k03-b-capex-spread-sensitivitaets-heatmap_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

*Direkt übernommen aus NB03 Chart 8 — hier eingebettet für eigenständige Ausführbarkeit.*  
Break-Even-Jahre als Funktion von [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) [EUR/kWh] und Intra-Tag-Spread [EUR/MWh].  
Horizontale Linien = CAPEX-Lernkurve-Projektionsjahre; vertikale Linie = aktueller CH-Spread.


In [ ]:
# ── Chart 8: CAPEX × Spread Sensitivitäts-Heatmap ───────────────────────────
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches  # Re-Import mit lokalem Alias (Legende)
# Parameter-Raum
capex_range  = np.arange(50, 501, 25)   # EUR/kWh
spread_range = np.arange(5, 101, 5)     # EUR/MWh Intra-Tag-Spread

# Batterieparameter Referenz: Privat 10kWh/5kW
CAPACITY_KWH  = 10
POWER_KW      = 5
EFFICIENCY    = CFG['pflicht']['simulation']['efficiency_roundtrip']
OPEX_RATE_C8  = CFG['pflicht']['wirtschaftlichkeit']['opex_rate']
LIFETIME_C8   = CFG['pflicht']['wirtschaftlichkeit']['lifetime_j']
DISPATCH_DAYS  = 200   # Dispatch-Tage/Jahr (aus Simulation NB02)
DISPATCH_H_DAY = 8     # Ø Dispatch-Stunden pro Tag (Laden + Entladen, aus NB02)
DISPATCH_HOURS = DISPATCH_DAYS * DISPATCH_H_DAY  # Gesamt-Dispatch-Stunden/Jahr

# Break-Even Berechnung für alle CAPEX × Spread Kombinationen
BE_matrix = np.zeros((len(capex_range), len(spread_range)))
for ci, capex_kwh in enumerate(capex_range):
    capex_total = capex_kwh * CAPACITY_KWH
    opex_annual = capex_total * OPEX_RATE_C8
    for si, spread_mwh in enumerate(spread_range):
        gross = spread_mwh * POWER_KW * EFFICIENCY * DISPATCH_HOURS / 1000  # DISPATCH_HOURS = Tage × h/Tag
        net   = gross - opex_annual
        BE_matrix[ci, si] = capex_total / net if net > 0 else 99
BE_plot = np.clip(BE_matrix, 0, 30)

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
ax.tick_params(colors=C_TICK)
for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)

cmap = LinearSegmentedColormap.from_list('be',
    [C_LOAD, C_PRICE, C_UTIL, '#4a0000'], N=256)
im = ax.imshow(BE_plot, aspect='auto', origin='lower', cmap=cmap,
               extent=[spread_range[0], spread_range[-1],
                       capex_range[0],  capex_range[-1]],
               vmin=0, vmax=30)
cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Break-Even [Jahre] (>25J = rot)', color='white')
cbar.ax.tick_params(colors='white')

# Konturlinien
CS = ax.contour(spread_range, capex_range, BE_plot,
                levels=[8, 12, 15, 20], colors='white', linewidths=1.2, alpha=0.8)
ax.clabel(CS, fmt='%dJ', colors='white', fontsize=FS_TICK)

# Aktueller Spread aus spread_zeitreihe.csv (falls vorhanden)
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')
current_spread = 20  # Fallback-Schätzung
if os.path.exists(SPREAD_FILE):
    _df_sp = pd.read_csv(SPREAD_FILE)
    current_spread = _df_sp['spread_median'].median()

# ── Marker: Heute (Spread × CAPEX) ────────────────────────────────────────────
today_spread = current_spread
today_capex  = 350
ax.plot(today_spread, today_capex,
        marker='*', color='white', markersize=18, zorder=10,
        label=f'Heute (~{today_spread:.0f} EUR/MWh, ~{today_capex} EUR/kWh)')
ax.annotate(f'Heute\n~{today_spread:.0f} EUR/MWh\n~{today_capex} EUR/kWh',
            xy=(today_spread, today_capex),
            xytext=(today_spread + 5, today_capex + 30),
            color='white', fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#222', alpha=0.7,
                      edgecolor='white', linewidth=1))

# ── Marker: Trigger ────────────────────────────────────────────────────────────
trig_spread = SPREAD_TRIGGER_EUR  # aus ../sync/config.json (SSOT)
trig_capex  = CAPEX_TRIGGER        # aus ../sync/config.json (SSOT)
ax.plot(trig_spread, trig_capex,
        marker='D', color=C_LOAD, markersize=11, zorder=10,
        label=f'Trigger ({trig_spread} EUR/MWh, {trig_capex} EUR/kWh)')
ax.annotate(f'Trigger\n{trig_spread} EUR/MWh\n{trig_capex} EUR/kWh',
            xy=(trig_spread, trig_capex),
            xytext=(trig_spread + 5, trig_capex - 60),
            color=C_LOAD, fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_LOAD, lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3', facecolor=C_LEGENDE_BG, alpha=0.7,
                      edgecolor=C_LOAD, linewidth=1))

# ── CAPEX-Lernkurve ────────────────────────────────────────────────────────────
ax.axhline(today_capex, color=C_PRIV, lw=LW, linestyle=':')
ax.text(spread_range[-1]*0.98, today_capex + 5,
        f'Heute ~{today_capex} EUR/kWh',
        color=C_PRIV, fontsize=FS_LEGENDE, ha='right')
for yrs, label in [(3, '-30% (3J)'), (5, '-50% (5J)')]:
    proj = today_capex * (0.90 ** yrs)
    ax.axhline(proj, color=C_PRIV, lw=1, linestyle=':', alpha=0.5)
    ax.text(spread_range[-1]*0.98, proj + 5, f'{label}: ~{proj:.0f} EUR/kWh',
            color=C_PRIV, fontsize=7.5, ha='right', alpha=0.8)

# ── Aktueller Spread (vertikale Linie) ────────────────────────────────────────
ax.axvline(today_spread, color='white', lw=LW, linestyle='--', alpha=0.6)

ax.set_xlabel('Intra-Tag-Spread [EUR/MWh]', color=C_ACHSE)
ax.set_ylabel('CAPEX [EUR/kWh]', color=C_ACHSE)
ax.set_title(
    f'Break-Even-Jahre: CAPEX \u00d7 Spread-Sensitivit\u00e4t'
    f' (Privat {CAPACITY_KWH}kWh/{POWER_KW}kW, {DISPATCH_DAYS} Dispatch-Tage/J)\n'
    'Gr\u00fcn = kurz amortisiert \u00b7 Rot = nie \u00b7 Konturlinien = Break-Even Jahre',
    color='white', fontsize=11)
ax.legend(fontsize=8.5, framealpha=0.5, facecolor=C_LEGENDE_BG,
          labelcolor='white', loc='lower right')

plt.tight_layout()
p8 = os.path.join(CHARTS_DIR, 'kuer_k03_sensitivitaet_heatmap.png')
plt.savefig(p8, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f'Chart 8 gespeichert: {p8}')

---
## 5. Synthese: Wann kippt der Business Case? <a id='synthese-wann-kippt-der-business-case_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

Die Sektionen 2–5 zeigen drei unabhängige Hebel, die den Business Case für Privatarbitrage kippen:

### Monitoring-Trigger

| Auslöser | Aktueller Wert (2023/2024) | Trigger-Schwelle | Quelle |
|----------|---------------------------|-----------------|--------|
| **Intra-Tag-Spread CH** (Monatsmedian) | ~24 EUR/MWh | **> 30 EUR/MWh dauerhaft** | `config.json → kuer.markt.spread_breakeven_privat_eur_mwh` |
| **CAPEX Privat** (Systempreis installiert) | ~350–400 EUR/kWh | **< 250 EUR/kWh** | `config.json → kuer.markt.capex_ziel_privat_eur_kwh` |

### Zeitliche Einordnung

```
2024 (heute):  Spread ~24 EUR/MWh · CAPEX ~370 EUR/kWh  →  Break-Even Privat > 25 J  ❌
2027 (Prognose): CAPEX ~260 EUR/kWh (−10%/J)            →  Break-Even ~18 J          ⚠️
2030 (NREL ATB Advanced): CAPEX ~190 EUR/kWh             →  Break-Even ~13 J          ⚠️
2030 + Spread 30 EUR/MWh: beides gleichzeitig            →  Break-Even ~9 J           ✅
2030 + VPP/FCR:           Revenue Stacking (+50 EUR/kWh/J) →  Break-Even ~5–7 J       ✅✅
```

### Kernaussage

**Kein einzelner Hebel reicht** — aber zwei Hebel zusammen (sinkende CAPEX + steigender Spread) ergeben den Kipppunkt. Der dritte Hebel (Revenue Stacking via FCR/VPP) macht das System auch bei heute herrschenden Marktbedingungen attraktiv, sobald Swissgrid den Flexibilitätsmarkt öffnet (~2026–2028).

→ Strategische Einordnung aller drei Hebel: [K_00 – Business Strategy](K_00_Business_Strategy.ipynb)


---
## Fazit <a id='fazit_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

Der Business Case ist stark von zwei gegenläufigen Trends geprägt:
**sinkenden CAPEX-Kosten** (Lernkurve) und dem **Preis-Spread** am Grosshandelsmarkt.
Die Sensitivitäts-Heatmap zeigt, dass insbesondere Utility- und Industrie-Segmente
bei Fortsetzung des Trends voraussichtlich innerhalb weniger Jahre die Break-Even-
Linie überschreiten.

Empfehlung: Monitoring-Trigger jährlich überprüfen (Spread-Index +
Market-CAPEX). Die konkreten Zeitpunkte werden in K_00 (Business Strategy)
zusammengeführt.


---
## Abschluss <a id='abschluss_K_03'></a>

[↑ Inhaltsverzeichnis](#toc_K_03)

Ausgabedateien und Transfer-Daten validieren.


In [ ]:
# ── Abschlusskontrolle K_03 ─────────────────────────────────────────────────
print('K_03 – Abschlusskontrolle')
print('=' * 60)
_charts=[f for f in os.listdir(CHARTS_DIR) if f.startswith('kuer_k03_')] if os.path.exists(CHARTS_DIR) else []
for _f in sorted(_charts): print(f'  ✅  {_f}')
print('→ Weiter mit nächstem Notebook.')



| [← K_02 – Cross-Border-Analyse](K_02_Cross_Border.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_04 – Saisonale Animationen →](K_04_Animationen.ipynb) |
|:---|:---:|---:|